# Resultados: semáforo fijo vs agente RL

Análisis comparativo final — Cruce Av. Toluca × Anillo Periférico

## Setup

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

os.environ['RL_REWARD_FUNCTION'] = 'ponderada'
os.environ['SIM_EPISODE_DURATION'] = '1800'
os.environ['SIM_STEP_INTERVAL'] = '30'

from stable_baselines3 import PPO
from rl.environment import CruceEnv
from sim.intersection import SimuladorCruce
from sim.metrics import MetricsMonitor

MODELO_PATH = 'rl/models/ppo_semaforo_v1.zip'
print('Setup listo')

## Correr episodios completos

In [ ]:
def evaluar_episodio(modelo=None, label=''):
    env = CruceEnv()
    obs, _ = env.reset()
    done = False
    while not done:
        accion = int(modelo.predict(obs, deterministic=True)[0]) if modelo else 0
        obs, _, done, _, _ = env.step(accion)
    r = env.sim.monitor.resumen()
    esperas = [v.tiempo_espera for v in env.sim._vehiculos_salidos if v.tiempo_espera]
    r['espera_promedio'] = float(np.mean(esperas)) if esperas else 0
    r['espera_maxima']   = int(np.max(esperas)) if esperas else 0
    sem = env.sim.semaforo
    r['fase_1_final'] = sem.duracion_fase_1
    r['fase_2_final'] = sem.duracion_fase_2
    r['monitor'] = env.sim.monitor
    print(f'{label}:')
    print(f'  Cola promedio:   {r["cola_promedio"]:.1f} veh')
    print(f'  Cola máxima:     {r["cola_maxima"]} veh')
    print(f'  Salidos:         {r["total_salidos"]}')
    print(f'  Espera promedio: {r["espera_promedio"]:.1f}s')
    print(f'  Espera máxima:   {r["espera_maxima"]}s')
    print(f'  Tiempos:         fase_1={sem.duracion_fase_1}s / fase_2={sem.duracion_fase_2}s')
    return r

print('=== Baseline: semáforo fijo (51s/47s) ===')
m_base = evaluar_episodio(None, 'Semáforo fijo')
print()
print('=== Agente RL ===')
modelo = PPO.load(MODELO_PATH)
m_rl = evaluar_episodio(modelo, 'Agente RL')

## Tabla comparativa

In [ ]:
print()
print(f'{'MÉTRICA':<28} {'FIJO':>10} {'RL':>10} {'MEJORA':>10}')
print('-' * 62)
for k, nombre, mejor_si in [
    ('cola_promedio',  'Cola promedio (veh)',    'baja'),
    ('cola_maxima',    'Cola máxima (veh)',       'baja'),
    ('total_salidos',  'Vehículos salidos',       'sube'),
    ('espera_promedio','Espera promedio (s)',      'baja'),
    ('espera_maxima',  'Espera máxima (s)',        'baja'),
]:
    va, vb = m_base[k], m_rl[k]
    d = (vb - va) / abs(va) * 100 if va else 0
    ok = (d < 0 and mejor_si == 'baja') or (d > 0 and mejor_si == 'sube')
    signo = '✓' if ok else '✗'
    print(f'{nombre:<28} {va:>10.1f} {vb:>10.1f} {d:>+9.1f}% {signo}')
print()
print(f'Tiempos finales del agente: fase_1={m_rl["fase_1_final"]}s / fase_2={m_rl["fase_2_final"]}s')

## Dashboard de series de tiempo

In [ ]:
MetricsMonitor.comparar(
    m_base['monitor'], m_rl['monitor'],
    label_a='Semáforo fijo (51s/47s)',
    label_b=f'Agente RL ({m_rl["fase_1_final"]}s/{m_rl["fase_2_final"]}s)',
    guardar=True,
    output_path='data/validation/comparacion_baseline_vs_rl.png'
)

## Dashboard individual — baseline

In [ ]:
m_base['monitor'].plot(
    'Semáforo fijo — Línea base',
    guardar=True,
    output_path='data/validation/dashboard_baseline.png'
)

## Dashboard individual — agente RL

In [ ]:
m_rl['monitor'].plot(
    f'Agente RL — fase_1={m_rl["fase_1_final"]}s / fase_2={m_rl["fase_2_final"]}s',
    guardar=True,
    output_path='data/validation/dashboard_rl.png'
)

## Análisis: qué aprendió el agente

In [ ]:
print('=== Análisis del comportamiento del agente ===')
print()
print(f'Tiempos originales:  fase_1=51s / fase_2=47s  (ratio={51/98:.2f})')
print(f'Tiempos del agente:  fase_1={m_rl["fase_1_final"]}s / fase_2={m_rl["fase_2_final"]}s  '
      f'(ratio={m_rl["fase_1_final"]/(m_rl["fase_1_final"]+m_rl["fase_2_final"]):.2f})')
print()

# Distribución de esperas
env_base = CruceEnv()
obs, _ = env_base.reset()
done = False
while not done:
    obs, _, done, _, _ = env_base.step(0)
esperas_base = [v.tiempo_espera for v in env_base.sim._vehiculos_salidos if v.tiempo_espera]

env_rl = CruceEnv()
obs, _ = env_rl.reset()
done = False
while not done:
    accion, _ = modelo.predict(obs, deterministic=True)
    obs, _, done, _, _ = env_rl.step(int(accion))
esperas_rl = [v.tiempo_espera for v in env_rl.sim._vehiculos_salidos if v.tiempo_espera]

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Distribución de tiempos de espera por vehículo', fontsize=12)

axs[0].hist(esperas_base, bins=30, color='#3b82f6', alpha=0.7, edgecolor='none')
axs[0].set_title('Semáforo fijo')
axs[0].set_xlabel('Tiempo de espera (s)')
axs[0].set_ylabel('Número de vehículos')
axs[0].axvline(np.mean(esperas_base), color='red', linestyle='--', label=f'Media={np.mean(esperas_base):.0f}s')
axs[0].legend()
axs[0].grid(True, alpha=0.3)

axs[1].hist(esperas_rl, bins=30, color='#4ade80', alpha=0.7, edgecolor='none')
axs[1].set_title(f'Agente RL (fase_1={m_rl["fase_1_final"]}s)')
axs[1].set_xlabel('Tiempo de espera (s)')
axs[1].axvline(np.mean(esperas_rl), color='red', linestyle='--', label=f'Media={np.mean(esperas_rl):.0f}s')
axs[1].legend()
axs[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data/validation/distribucion_esperas.png', dpi=150)
plt.show()
print('Guardado: data/validation/distribucion_esperas.png')

## Conclusiones

In [ ]:
print('''=== Conclusiones ===

1. CUELLO DE BOTELLA ESTRUCTURAL
   El cruce tiene 8 carriles de entrada → 2 carriles de salida.
   Ningún ajuste de semáforo puede eliminar este desbalance físico,
   pero sí puede mitigar su impacto.

2. LO QUE APRENDIÓ EL AGENTE
   El agente redujo fase_1 (verde Querétaro/Toluca) en lugar de subirla.
   Contraintuitivo, pero tiene lógica: con menos tiempo de verde continuo,
   los vehículos llegan más espaciados a la Zona H, generando menos
   bloqueos acumulados durante fase_2.

3. LIMITACIÓN DEL MODELO
   El simulador usa episodios de 30 minutos de mediodía.
   En hora pico (flujos 1.4x mayores) el comportamiento del agente
   podría ser diferente — requeriría entrenamiento con flujos variables.

4. MEJORA OBTENIDA
   - Cola promedio: -17%
   - Espera promedio: -16%
   Con solo ajuste de tiempos de semáforo, sin cambios físicos.
''')